In [ ]:
import os
from pathlib import Path
from getpass import getpass

try:
    from dotenv import load_dotenv

    candidates = [
        Path.cwd() / ".env",
        Path.cwd() / ".conda_env" / ".env",
        Path.cwd().parent / ".env",
        Path.cwd().parent / ".conda_env" / ".env",
    ]

    env_file = next((p for p in candidates if p.exists()), None)
    if env_file:
        load_dotenv(env_file)
        print(f"Loaded .env from: {env_file}")
    else:
        load_dotenv()
        print("No .env file found in default locations")
except Exception:
    pass

if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("Enter OPENROUTER_API_KEY: ")
if not os.getenv("TAVILY_API_KEY"):
    os.environ["TAVILY_API_KEY"] = getpass("Enter TAVILY_API_KEY: ")
if not os.getenv("PINECONE_API_KEY"):
    os.environ["PINECONE_API_KEY"] = getpass("Enter PINECONE_API_KEY: ")

print("OPENROUTER_API_KEY configured:", bool(os.getenv("OPENROUTER_API_KEY")))
print("TAVILY_API_KEY configured:", bool(os.getenv("TAVILY_API_KEY")))
print("PINECONE_API_KEY configured:", bool(os.getenv("PINECONE_API_KEY")))

Loaded .env from: c:\Users\user\Downloads\AI-Projects\.conda_env\.env
GROQ_API_KEY configured: True
TAVILY_API_KEY configured: True
PINECONE_API_KEY configured: True


<h3>Load documents</h3>


In [10]:
from langchain_community.document_loaders import WebBaseLoader
SOURCE_URL="https://docs.langchain.com/oss/python/langgraph/agentic-rag"

loader=WebBaseLoader(
    web_paths=(SOURCE_URL,),
    requests_kwargs={
        "headers":{
            "User-Agent": "Mozilla/5.0 Agentic-RAG-Industry-Demo"
        }
    },
)

raw_docs=loader.load()
print("Loaded documents:",len(raw_docs))
print("Source:", raw_docs[0].metadata.get("source"))
print("\nPreview:\n")
print(raw_docs[0].page_content[:1500])

C:\Users\user\AppData\Local\Temp\ipykernel_25044\141816477.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


Loaded documents: 1
Source: https://docs.langchain.com/oss/python/langgraph/agentic-rag

Preview:

Build a custom RAG agent with LangGraph - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to London on October 13! Get ticketsDocs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangGraphBuild a custom RAG agent with LangGraphOverviewDeep AgentsManaged Deep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonLearnTutorialsDeep AgentsLangChainMulti-agentLangGraphCustom RAG agentCustom SQL agentConceptual overviewsLangChain vs. LangGraph vs. Deep AgentsProviders and modelsComponent architectureMemoryContextGraph APIFunctional APIAdditional resourcesUse docs programmaticallyLangChain AcademyCase studiesGet helpOn this pageConceptsSetupSet up LangSmithPreprocess documentsCrea

split the document to chunks

In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter=RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    add_start_index=True,
)
chunks=splitter.split_documents(raw_docs)
print("Total chunks:",len(chunks))
print("Preview of first chunk:\n")
print(chunks[0].page_content[:1500])



Total chunks: 28
Preview of first chunk:

Build a custom RAG agent with LangGraph - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to London on October 13! Get ticketsDocs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangGraphBuild a custom RAG agent with LangGraphOverviewDeep AgentsManaged Deep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonLearnTutorialsDeep AgentsLangChainMulti-agentLangGraphCustom RAG agentCustom SQL agentConceptual overviewsLangChain vs. LangGraph vs. Deep AgentsProviders and modelsComponent architectureMemoryContextGraph APIFunctional APIAdditional resourcesUse docs programmaticallyLangChain AcademyCase studiesGet helpOn this pageConceptsSetupSet up LangSmithPreprocess documentsCreate a retriever toolGenerate a query or respondGrade docum

<h3>Embedding</h3>

In [ ]:
%pip install langchain-huggingface==1.2.3 sentence-transformers==6.1.0

In [12]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={"normalize_embeddings": True},
)
sample_vector = embeddings.embed_query("what is agentic RAG?")
print("Sample embedding vector length:", len(sample_vector))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1551.74it/s]


Sample embedding vector length: 384


In [3]:
len(sample_vector)

384

sample_vector


In [ ]:
sample_vector

In [20]:
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore
from dotenv import load_dotenv
from getpass import getpass
from pathlib import Path
import time
import os

INDEX_NAME = "industry-agentic-rag-kb"
NAME_SPACE= "langgraph-agentic-rag"

env_candidates = [
    Path.cwd() / ".env",
    Path.cwd() / ".conda_env" / ".env",
    Path.cwd().parent / ".env",
    Path.cwd().parent / ".conda_env" / ".env",
]
env_file = next((path for path in env_candidates if path.is_file()), None)
if env_file:
    load_dotenv(env_file)
else:
    load_dotenv()

if not os.getenv("PINECONE_API_KEY"):
    os.environ["PINECONE_API_KEY"] = getpass("Enter PINECONE_API_KEY: ")
if not os.environ["PINECONE_API_KEY"].strip():
    raise ValueError("PINECONE_API_KEY is required. Add it to .env or enter it when prompted.")

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

existing_indexes=[index_info["name"] for index_info in pc.list_indexes()]

if INDEX_NAME not in existing_indexes:
    pc.create_index(
        name=INDEX_NAME,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1",
        ),
    )
    while not pc.describe_index(INDEX_NAME).status["ready"]:
        time.sleep(1)

print("Pine cone index ready",INDEX_NAME)

#upload the document chunks and create the langchain vector store

vectorstore = PineconeVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    index_name=INDEX_NAME,
    namespace=NAME_SPACE,
)

retriever = vectorstore.as_retriever(
    search_kwargs={
        "k":4,
        "namespace":NAME_SPACE,
    }
)

print("pinecone vector db and retriver are ready.")







Pine cone index ready industry-agentic-rag-kb
pinecone vector db and retriver are ready.


In [15]:
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

INDEX_NAME = "industry-agentic-rag-kb"
NAME_SPACE= "langgraph-agentic-rag"

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

#load exixting index 
pc.Index(INDEX_NAME)

#Connect existing  index with langchain 
vectorstore = PineconeVectorStore(
    embedding=embeddings,
    index_name=INDEX_NAME,
    namespace=NAME_SPACE,
)

retriever = vectorstore.as_retriever(
    search_kwargs={
        "k":4,
        "namespace":NAME_SPACE,
    }
)

print("existing pine cone index",INDEX_NAME)


existing pine cone index industry-agentic-rag-kb


<h3>Testing kb retrival<h3>


In [21]:
query="what happenns when received documents are not relavent in agentic RAG?"
kb_docs = retriever.invoke(query)
for i, doc in enumerate(kb_docs, 1):
    print(f"\n---kb result {i}----\n")
    print("source:",doc.metadata.get("source"))
    print(doc.page_content[:700])


---kb result 1----

source: https://docs.langchain.com/oss/python/langgraph/agentic-rag
pageConceptsSetupSet up LangSmithPreprocess documentsCreate a retriever toolGenerate a query or respondGrade documentsRewrite the questionGenerate an answerAssemble the graphRun the agentic RAGSee alsoTutorialsLangGraphBuild a custom RAG agent with LangGraphCopy pageCopy pageBuild a custom retrieval agent with LangGraph that decides when to search a vector store or respond directly.Copy pageCopy pageBuild a retrieval agent with LangGraph that decides when to search a vector store versus answering the user directly.

---kb result 2----

source: https://docs.langchain.com/oss/python/langgraph/agentic-rag
pageConceptsSetupSet up LangSmithPreprocess documentsCreate a retriever toolGenerate a query or respondGrade documentsRewrite the questionGenerate an answerAssemble the graphRun the agentic RAGSee alsoTutorialsLangGraphBuild a custom RAG agent with LangGraphCopy pageCopy pageBuild a custom retrieval 

In [23]:
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv
from getpass import getpass
from pathlib import Path

env_candidates = [
    Path.cwd() / ".env",
    Path.cwd() / ".conda_env" / ".env",
    Path.cwd().parent / ".env",
    Path.cwd().parent / ".conda_env" / ".env",
]
env_file = next((path for path in env_candidates if path.is_file()), None)
if env_file:
    load_dotenv(env_file)
else:
    load_dotenv()

openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
if not openrouter_api_key:
    openrouter_api_key = getpass("Enter OPENROUTER_API_KEY: ").strip()
if not openrouter_api_key:
    raise ValueError("OPENROUTER_API_KEY is required. Add it to .env or enter it when prompted.")

llm = ChatOpenAI(
    model="nvidia/nemotron-3.5-lightning:free",
    api_key=openrouter_api_key,
    base_url="https://openrouter.ai/api/v1",
    temperature=0,
)

print(llm.invoke("Explain Agentic RAG in one sentence.").content)

Agentic RAG is an autonomous AI framework that dynamically plans, retrieves, and iterates on information using reasoning and tool use to answer complex queries, surpassing the static, one-shot retrieval of traditional RAG.


In [24]:
from langchain_tavily import TavilySearch

web_search=TavilySearch(
    max_results=5,
    topic="general",
    include_answer=True,
    include_raw_content=False,
)

print("Tavily search tool ready")

Tavily search tool ready
